# Google Keywords · Palavras-chave do Google

38 million searches with volume, CPC and competition (Brazil, US, Spain, Argentina), 1.73 million search patterns of five algorithms and the labels the author gave them while choosing niches.

Data: [Kaggle](https://www.kaggle.com/datasets/caiquedourado/google-keywords-dataset) · Code and method: [GitHub](https://github.com/CaiqueDourado/google-keywords-dataset). On GitHub the notebook runs on `sample/`.

In [1]:
%pip install -q duckdb  # already present on most images
import glob, os
import duckdb

# Kaggle: full Parquet files; GitHub: the CSV samples
found = glob.glob('/kaggle/input/**/palavras_chave-*.parquet', recursive=True)
if found:
    base = os.path.dirname(found[0])
    KW = f"read_parquet('{base}/palavras_chave-*.parquet')"
    PAD = f"read_parquet('{base}/padroes.parquet')"
    ALG = f"read_parquet('{base}/padroes_algoritmos.parquet')"
    QUE = f"read_parquet('{base}/perguntas.parquet')"
else:
    base = '../sample' if os.path.exists('../sample') else 'sample'
    KW = f"read_csv_auto('{base}/palavras_chave_sample.csv')"
    PAD = f"read_csv_auto('{base}/padroes_sample.csv')"
    ALG = f"read_csv_auto('{base}/padroes_algoritmos_sample.csv')"
    QUE = f"(SELECT * FROM {KW} WHERE mercado = 'BR' AND regexp_matches(lower(palavra), '^(como|qual|onde|quando|quanto|por que|melhor)'))"
q = lambda sql: duckdb.sql(sql).df()
CLEAN = 'NOT adulto' if found else 'true'   # adult searches are flagged in the full data; the samples have none
print('full data' if found else 'sample', base)

Note: you may need to restart the kernel to use updated packages.


sample ../sample


## Searches by market and source

In [2]:
q(f"""SELECT mercado, fonte, count(*) buscas, count(volume) com_volume, sum(volume) volume_total,
             round(median(cpc) FILTER (cpc > 0), 2) cpc_mediano
      FROM {KW} GROUP BY ALL ORDER BY buscas DESC""")

,mercado,fonte,buscas,com_volume,volume_total,cpc_mediano
0,US,dataforseo,2500,2496,11527730.0,0.78
1,AR,dataforseo,2500,2500,3974930.0,0.04
2,ES,dataforseo,2500,2461,3355790.0,0.23
3,BR,ubersuggest_2018,2387,1396,265820.0,1.07
4,BR,dataforseo,113,113,151450.0,0.18


## The biggest searches in Brazil

In [3]:
q(f"SELECT palavra, volume, concorrencia, cpc FROM {KW} WHERE mercado = 'BR' AND {CLEAN} ORDER BY volume DESC NULLS LAST LIMIT 20")

,palavra,volume,concorrencia,cpc
0,tabela gauchão 2024,27100,0.00,0.00
1,resultado do jogo do bicho para todos bahia de...,18100,0.00,0.00
2,fiat pulse preço,18100,0.19,0.17
3,casas bahia eletrodomestico,12100,0.10,7.00
4,cupcakes de chocolate,9900,0.03,2.00
5,poupatempo sbc,9900,0.10,0.01
6,histórias em quadrinhos,9900,0.02,0.72
7,remédio para rouquidão,8100,0.79,0.01
8,jornal o noroeste,6600,0.00,0.00
9,conjunto de time,6600,0.99,0.08


## Questions people ask

Searches that start with como, qual, onde, quando, quanto, por que, melhor…

In [4]:
q(f"SELECT palavra, volume FROM {QUE} ORDER BY volume DESC NULLS LAST LIMIT 20")

,palavra,volume
0,como achar as redes sociais de uma pessoa pela...,1300
1,como fazer o cha de anis estrelado,390
2,como fazer peso de porta de flor,320
3,como abrir cnpj gratuito,320
4,como sacar o lis do itau,320
5,onde comprar ginseng,320
6,como colocar senha nas fotos iphone,210
7,como se joga bolinha de gude,210
8,como usar olho de ricino no cabelo,210
9,quantos anos o saiko tem,210


## Patterns of each algorithm

Viana takes prefixes, Vilela runs of words; Barbosa, Figueredo and Rosa do the same after replacing entities by `[type]`.

In [5]:
q(f"""SELECT algoritmo, count(*) padroes, sum(tem_entidade::int) com_entidade, count(rotulos_uso) rotulados
      FROM {ALG} GROUP BY 1 ORDER BY padroes DESC""")

,algoritmo,padroes,com_entidade,rotulados
0,barbosa,400,400.0,107
1,rosa,400,400.0,0
2,viana,400,0.0,203
3,vilela,400,0.0,42
4,figueredo,400,400.0,78


In [6]:
q(f"""SELECT * FROM (SELECT algoritmo, padrao, busca_lider, num_buscas, volume_total,
             row_number() OVER (PARTITION BY algoritmo ORDER BY volume_total DESC) r FROM {ALG})
      WHERE r <= 3 ORDER BY algoritmo, r""")

,algoritmo,padrao,busca_lider,num_buscas,volume_total,r
0,barbosa,[site] entrar,´ facebook entrar,1875,165143090,1
1,barbosa,[local] [local],detran pe,726128,61628180,2
2,barbosa,[produto] [produto],afinador de violao,501026,36458170,3
3,figueredo,surpres aniversari [site],roda de surpresas do aniversário do google,19,673000,1
4,figueredo,[cidade] diret [site],facebook entrar agora direto no meu facebook,17,563910,2
5,figueredo,with [site],login with hotmail,294,562510,3
6,rosa,par baix [site],aplicativos para baixar videos do youtube,1622,1697980,1
7,rosa,com vide [site],como baixar video do youtube,10569,1508540,2
8,rosa,sao [cidade],sao carlos agora,35588,1215750,3
9,viana,facebook entrar,´ facebook entrar,1023,251104940,1


## Niches by curation label

Labels the author gave while choosing sites: how the niche makes money and the kind of page it needs.

In [7]:
q(f"""SELECT trim(r) rotulo, count(*) padroes FROM (SELECT unnest(string_split(rotulos_uso, ',')) r FROM {ALG})
      GROUP BY 1 ORDER BY padroes DESC LIMIT 20""")

,rotulo,padroes
0,AdSense,398
1,Afiliados,157
2,App,107
3,Produtos,68
4,Previsão do Tempo,37
5,Resultado do Jogo do Bicho,34
6,Entrar em Sites,24
7,Esportes,22
8,Filmes,18
9,TV,16


In [8]:
# affiliate niches with an entity, biggest first
q(f"""SELECT padrao, busca_lider, num_buscas, volume_total, rotulos_uso FROM {ALG}
      WHERE rotulos_uso LIKE '%Afiliados%' AND tem_entidade AND {CLEAN} ORDER BY volume_total DESC LIMIT 20""")

,padrao,busca_lider,num_buscas,volume_total,rotulos_uso
0,com faz [comida],como fazer panqueca,98097,12142260,"AdSense, Afiliados, Como Fazer, Receitas"
1,baix vide [site],baixar video do youtube,1587,11083930,"AdSense, Afiliados, App, Como baixar Video"
2,baix [music],baixar música,11365,10075640,"AdSense, Afiliados, App, Como baixar Música"
3,[loja/loja-virtual] [produto],dafiti calçados,18196,4415440,"AdSense, Afiliados, Produtos, Produtos em Site..."
4,[medicamento] par,remédio para emagrecer,18031,4201150,"Afiliados, Medicamento, Medicamento para, Prod..."
5,baix music [site],baixar musica do youtube,528,3755460,"AdSense, Afiliados, App, Como baixar Música"
6,[produto] concurs,pci concursos,1684,3504780,"AdSense, Afiliados, App, Concurso Público"
7,prec [produto],preço de celular,65140,3477510,"Adsense, Afiliados, Preço de Produtos, Produtos"
8,[acessorios-de-informatica] concurs,pci concursos,802,3430710,"AdSense, Afiliados, App, Concurso Público"
9,[placa-de-video] concurs,pci concursos,800,3430640,"AdSense, Afiliados, App, Concurso Público"


## Prefix patterns (SuperNichos)

The table shown in the SuperNichos demo: patterns of stemmed words, with 1–20 bands for volume, competition and CPC.

In [9]:
q(f"SELECT padrao, mercado, num_buscas, volume_total, busca_lider, faixa_volume, faixa_concorrencia, faixa_cpc FROM {PAD} WHERE {CLEAN} ORDER BY volume_total DESC LIMIT 20")

,padrao,mercado,num_buscas,volume_total,busca_lider,faixa_volume,faixa_concorrencia,faixa_cpc
0,facebook entrar,BR,1027,183512650,´ facebook entrar,1,10,13
1,how to,US,56809,126761740,NaN,1,14,15
2,what is,US,42309,118532600,NaN,1,11,12
3,previsa temp,BR,92914,96124520,previsão do tempo,1,10,12
4,el tiemp,ES,15308,72065430,NaN,1,3,2
5,eight sleep,US,4,68013200,NaN,1,19,16
6,classific brasile,BR,922,64746090,classificação brasileirão série a,1,9,11
7,classific brasile seri,BR,295,62144660,classificação brasileirão série a,1,8,12
8,how mani,US,40573,60397260,NaN,1,2,8
9,whatsapp web,BR,312,58136810,whatsapp web,1,11,14


## CPC is on two scales

DataForSEO looks like US dollars; the 2018 Ubersuggest unit is unknown. Compare CPC only within one `fonte`.

In [10]:
q(f"""SELECT fonte, mercado, quantile_cont(cpc, [0.5, 0.9, 0.99]) p50_p90_p99 FROM {KW} WHERE cpc > 0
      GROUP BY ALL ORDER BY ALL""")

,fonte,mercado,p50_p90_p99
0,dataforseo,AR,"[0.04, 0.21, 0.8637999999999988]"
1,dataforseo,BR,"[0.18, 0.68, 2.425199999999999]"
2,dataforseo,ES,"[0.23, 0.9, 3.115]"
3,dataforseo,US,"[0.78, 2.908, 9.56939999999998]"
4,ubersuggest_2018,BR,"[1.07, 35.0, 118.71999999999935]"
